# GEMS Milestone 1: champion → validated submission (canonical notebook)

**One notebook, one configuration.** Recovers the strongest valid frozen W2 champion, reproduces the official evaluation, diagnoses export parity, applies explicit promotion gates, writes a validated GeoTIFF only when every mandatory gate passes, and records a manually submitted public score. Nothing is trained, re-inferred or uploaded.

This notebook **replaces** `00_ENVIRONMENT_AND_ARTIFACTS` … `04_LEADERBOARD_REVIEW` (kept for history, superseded). No workaround cell is needed: the shared grid loader reads only the declared grids.

| | |
|---|---|
| Runtime | CPU, **High-RAM** (no GPU needed) |
| Inputs (read-only) | `MyDrive/GEMS_2/outputs/p6_fast_track/{stack, boost/benchmark, recovery/export_parity}`, `experiments/b0_reference__d54e938a`, `experiments/b1_bg_weight__7755635b`, folds, template, M9 selection |
| Outputs | `MyDrive/GEMS_2/outputs/leaderboard_focus/m1_champion_recovery/` |
| Code | `gems_bench/leaderboard_focus/workflow.py` (`Runner`) |

| stage | cell | writes | status meaning |
|---|---|---|---|
| 0 | configuration, bootstrap | `environment/` | – |
| 1 | artifact inventory | `artifact_inventory.*`, `staging_manifest.json` | COMPLETE = every required artifact OK |
| 2 | frozen champion selection | `selection/` | COMPLETE = **selection** complete (not promotion) |
| 3 | official validation | `validation/` | COMPLETE / WARNING (caveats) / BLOCKED |
| 4 | export parity + promotion gates | `parity/`, `selection/promotion_gates.*` | READY_FOR_EXPORT or the blocking gate |
| 5 | submission export | `submissions/` | READY_FOR_UPLOAD only when every check passed |
| 6 | leaderboard review | `leaderboard/` | manual entries only |
| 7 | final report | `reports/M1_*.md` | executive summary, failure diagnostics, next action |

**Resume after a disconnect:** run Stage 0 again, then the stage cells. Finished stages print `SKIP_CACHED`; a BLOCKED stage is always re-evaluated.

## Stage 0 · Configuration (the only cell you edit)

`mode`: `FULL` (every stage, stops downstream on a blocker) · `RECOVER` (inventory + selection) · `VALIDATE` (validation + parity on the persisted selection) · `SUBMIT` (export the persisted validated recipe if ready). A stage outside the mode prints `NOT_REQUIRED` and leaves its persisted result untouched.

`M1_CONFIG` overrides `configs/leaderboard_focus/m1.yaml` (normally empty). The geographic floor is **mandatory** by default: a champion below 0.1758 is not exported. To export it anyway you must write the reason: `M1_CONFIG = {'promotion': {'geographic_gate': 'advisory', 'override_note': '<why>'}}` (recorded in every manifest).

In [ ]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',                       # pin the pushed commit SHA for a reproducible run
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',

    'mode': 'FULL',                      # FULL | RECOVER | VALIDATE | SUBMIT
    'reuse_cached': True,                # reuse staged files while the inventory fingerprint is unchanged
    'overwrite_existing': False,
    'strict_provenance': True,           # a changed B0 / B1 experiment id blocks M1

    'required_grids': ['boost_a'],       # the only grids read; unusable -> selection BLOCKED (no fallback)
    'train_cnn': False,                  # M1 never trains: True is refused
    'train_transformer': False,
    'run_extra_seeds': False,

    'save_figures': True,                # PNG files (their _data.csv is always written)
    'save_csv': True,                    # required (CSV tables are the stage contract)
    'save_parquet': True,
    'save_geotiff': True,                # False: stage 5 is NOT_REQUIRED
    'sync_to_drive': True,
}
PRIVATE_REPO = False
M1_CONFIG = {}                          # overrides of configs/leaderboard_focus/m1.yaml
EXPECT = {'code_rev': 'm1.2-r1', 'commit': None}   # set 'commit' to the pushed SHA
LEADERBOARD_ENTRY = {                   # stage 6: fill ONLY after a manual DrivenData upload
    'submission_id': '', 'submission_file': '', 'public_dti': None, 'submitted_at': '', 'notes': '',
}

## Stage 0 · Bootstrap: Drive, strict clone, version lock, runner

A ref that cannot be resolved stops here (no fallback branch). The runtime must be fresh (`gems_bench` not yet imported). The P6 engine is attached lazily, once, by the first stage that needs it.

In [ ]:
import base64, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = 'https://github.com/' + CONFIG['repo'] + '.git'

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(args[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + ref, check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'm1_colab_active', 'origin/' + ref)
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)       # a SHA / tag must resolve, else this raises
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: Runtime -> Restart session, then run from Stage 0')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab]'], check=True)
sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
import gems_bench.leaderboard_focus as LF
assert LF.M1_CODE_REV == EXPECT['code_rev'], f'code rev {LF.M1_CODE_REV} != {EXPECT["code_rev"]}'
assert not EXPECT['commit'] or HEAD.startswith(EXPECT['commit']), f'HEAD {HEAD} != {EXPECT["commit"]}'

import pandas as pd
from IPython.display import Markdown, display
from gems_bench.leaderboard_focus.workflow import Runner
R = Runner(CONFIG, HEAD, M1_CONFIG)
print('Requested', ref, '| SHA', HEAD, '| M1 code rev', LF.M1_CODE_REV, '| mode', R.nb['mode'], '| stages', R.stages)
print('Geographic gate:', R.cfg['promotion']['geographic_gate'], '| floor', R.cfg['selection']['geo_floor'],
      '| grids', R.cfg['champion']['grids'], '| outputs', R.cfg['outputs'])

## Stage 1 · Artifact inventory and selective staging

Every artifact in `m1.yaml → artifacts` is checked on Drive: existence, sha256, experiment identity (B0 `b0_reference__d54e938a`, B1 `b1_bg_weight__7755635b`), required file names (`oof.tif`, `disc0-2.tif`; W2 `geo_0-2.npy`, `disc_official_0-2.npy`). Extra files are accepted and listed. Only `staged: true` files are copied locally — never boost B/C/D, W1 seeds, checkpoints or historical submission rasters.

Statuses: `OK`, `MISSING`, `MISSING_REQUIRED_FILES`, `COUNT_MISMATCH`, `AMBIGUOUS`, `ID_CHANGED`. A required artifact that is not OK stops every later stage (they record INCOMPLETE).

In [ ]:
A = R.step('artifacts')
if 'inventory' in A:
    display(A['inventory'][['id', 'required', 'status', 'n_files', 'experiment_ids', 'extra_files', 'note']])

## Stage 2 · Frozen champion selection

Loads **only** `required_grids` (boost A) through `full_benchmark.load_grids(fg, sources=…, required=…)`; an absent, empty, header-only or malformed required grid makes the stage BLOCKED with the file facts in `selection/grid_provenance.csv`. Selection = the predeclared p6g rule (nested single policy, objective disc, worst zone ≥ 0.153414). COMPLETE here means *selection complete*: promotion is decided in Stage 4.

In [ ]:
S2 = R.step('selection')
W = R.light().ws
display(W.load_table('selection/grid_provenance.csv'))
C = W.load_table('selection/champion_candidates.csv')
if len(C):
    display(C[[c for c in ('candidate_id', 'role', 'status', 'eligible', 'disc_dti', 'disc_worst', 'geo_dti',
                           'geo_floor_met', 'decision') if c in C]])

## Stage 3 · Official validation

Per held-out zone the champion cell is rebuilt from the frozen maps and scored by the existing official scorer; components must equal the saved grid (1e-6). Recorded historical values are reproduced from the same grid and labelled by protocol: a `separate`-protocol geographic value (geo cell chosen on geo labels) is **not comparable** with the champion's single-policy geo DTI, so its gap is explained, not treated as a defect (`validation/protocol_comparison.csv`). The dry zone-stitched build checks template, coverage, per-zone budget and recipe.

In [ ]:
S3 = R.step('validation')
W = R.light().ws
display(W.load_table('validation/official_scores.csv'))
display(W.load_table('validation/protocol_comparison.csv'))

## Stage 4 · Export parity, then promotion gates

Parity: fold assignment, duplicated / missing pixels, zone score vs evaluated score (Spearman ≥ 0.999, strict), recipe identity; historical-export rows are documented and never block the champion. Then the promotion gates, each PASS / WARNING / BLOCKED / PENDING: `SELECTION_COMPLETE`, `DISCOVERY_ELIGIBLE`, `GEOGRAPHIC_GATE_PASS`, `VALIDATION_PASS`, `PARITY_PASS` → `READY_FOR_EXPORT`.

In [ ]:
S4 = R.step('parity')
W = R.light().ws
P = W.load_table('parity/parity_report.csv')
if len(P):
    display(P[P.scope == 'champion'][['check', 'unit', 'status', 'detail']])
G = R.step('promotion')
display(W.load_table('selection/promotion_gates.csv'))

## Stage 5 · Submission export

Runs only when promotion is READY_FOR_EXPORT; otherwise it records BLOCKED / INCOMPLETE and writes **no** file. The written band must equal the array validated in Stage 3 (sha256), the recipe must equal the Stage-2 recipe, and the GeoTIFF contract (1 band, float32, EPSG 32611, 100 m, values in [0, 1], ≤ 50 MB) must hold. Versioned name `submission_m1_<family>_<recipe_id>.tif`; never overwritten; never uploaded.

In [ ]:
S5 = R.step('submissions')
sub = R.light().ws.read_json('submissions/submission_manifest.json') or {}
print('SUBMISSION:', sub.get('status'), '|', sub.get('file'), '| sha256', sub.get('sha256'))
for r in sub.get('reasons') or []:
    print('  reason:', r)

## Stage 6 · Leaderboard review (manual)

1. Only when Stage 5 says `READY_FOR_UPLOAD`: download the file from Drive, check `sha256sum` against `submissions/submission_sha256.txt`, upload it by hand on DrivenData (competition 306; 3 scored submissions per rolling 7 days).
2. Fill `LEADERBOARD_ENTRY` in Stage 0 (`submitted_at` as `YYYY-MM-DD`) and run this cell. An entry for another file, a changed file or a duplicate id is refused.

Public scores are recorded for the history only: they never feed back into selection.

In [ ]:
S6 = R.step('leaderboard', entry=LEADERBOARD_ENTRY)
display(R.light().ws.load_table('leaderboard/leaderboard_history.csv'))

## Stage 7 · Final report

Written from the persisted manifests: `reports/M1_EXECUTIVE_SUMMARY.md`, `reports/M1_FAILURE_DIAGNOSTICS.md` (every non-passing stage, matched against the known failure modes), `reports/M1_NEXT_ACTION.md`, `reports/stage_status.csv`.

In [ ]:
REP = R.step('report')
display(Markdown(R.light().ws.path('reports/M1_EXECUTIVE_SUMMARY.md').read_text(encoding='utf-8')))
display(R.table())

## Verification checklist

1. Stage 1 `COMPLETE`; `b0_maps` / `b1_maps` show the recorded experiment ids and `oof_fold0-2.tif` as extras.
2. Stage 2 `COMPLETE`; `grid_provenance.csv` shows `boost_a` usable and nothing else loaded.
3. Stage 3 `COMPLETE` or `WARNING` (read every warning); `protocol_comparison.csv` explains each recorded value.
4. Stage 4 parity `COMPLETE`; promotion gates: read `GEOGRAPHIC_GATE_PASS` explicitly.
5. Stage 5 `READY_FOR_UPLOAD` before any upload; `export_parity_report.json → equal: true`.
6. `reports/M1_NEXT_ACTION.md` names the next step.

## Troubleshooting

See `reports/M1_FAILURE_DIAGNOSTICS.md` (generated) and `docs/leaderboard_focus/TROUBLESHOOTING.md`. Common cases: `required grid(s) unusable` → restore `outputs/p6_fast_track/boost/benchmark/grid.csv`; `produced by different stage code` → run mode `FULL`; `GEOGRAPHIC_GATE_PASS BLOCKED` → a decision (keep blocked, or advisory + override note); `gems_bench already imported` → restart the session.

Milestone 1 is complete only after a `READY_FOR_UPLOAD` file exists and you have verified it. Milestone 2 is not started here.